In [6]:
!pip install chromadb
!pip install -U -q "google-genai"

In [7]:
from google.colab import userdata
from google import genai

# El cliente de Gemini para hacer los embedding
GEMINI_API_KEY = userdata.get('GOOGLE_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

## API de TMDB para conseguir información de películas y cast

In [8]:
import json
import requests

# Para obtener información de una película
# usamos la API de TMDB
TMDB_API_KEY = userdata.get('TMDB_KEY')
TMDB_HEADERS = {
      "accept": "application/json",
      "Authorization": f"Bearer {TMDB_API_KEY}"
}

def get_movies_info(title):
  url = f"https://api.themoviedb.org/3/search/movie?query={title}&include_adult=false&language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_reviews(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/reviews?language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_cast(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?language=en-US"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_person_details(person_id):
  url = f"https://api.themoviedb.org/3/person/{person_id}"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)


## Añadir películas a la colección "movies" de ChromaDB

In [9]:
import chromadb
chroma_client = chromadb.PersistentClient(path="chroma_db")

# Función para añadir películas encontradas a la colección
# la colección es una parte de la base de datos
# hay colecciones por categorías (películas, actores, reviews...)
def add_movies_to_collection(movies_info):
  movies_col = chroma_client.get_or_create_collection('movies')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  def get_cast_info(movie):
    cast_found = get_movie_cast(str(movie['id']))
    director = "";
    cast = "";

    for person in cast_found['cast']:
      if person['known_for_department'] == 'Directing':
        director += person['name'] + ", "
      else:
        cast += person['name'] + ", "
    return director, cast

  def get_metadata(movie, director, cast):
    return {
        "movie_title" : movie['title'],
        "director"    : director,
        "cast"        : cast,
        "popularity"  : movie['popularity'],
        "release_date": movie['release_date'],
        "vote_average": movie['vote_average'],
        "vote_count"  : movie['vote_count']
    }

  for movie in movies_info['results']:
    # Consultamos con nuestra colección
    result = movies_col.get(
      ids=[str(movie['id'])],
      include=[]
    )

    # Si no existe, la añade
    if not result['ids'] and movie['overview']:
      # Primero obtenemos al cast
      director, cast = get_cast_info(movie)

      content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                        contents=movie['overview']).embeddings[0]
      ids.append(str(movie['id']))
      embeddings.append(content_embeddings.values)
      metadatas.append(get_metadata(movie, director, cast))
      documents.append(movie['overview'])

  if len(ids) > 0:
    movies_col.add(
        ids = ids,
        embeddings = embeddings,
        metadatas = metadatas,
        documents = documents
    )

  print(f"Added {len(ids)} movies.")

# Ejemplo de cómo usarlo junto a la búsqueda en TMDB
movies = get_movies_info('mario the movie')
add_movies_to_collection(movies)

Added 5 movies.


## Añadir reviews a la colección "reviews" de ChromaDB

In [10]:
def add_reviews_to_collection(movie_id):
  reviews_col = chroma_client.get_or_create_collection('reviews')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  reviews = get_movie_reviews(movie_id)

  def get_metadata(review):
    if review["author_details"]["rating"]:
      return {
          "author"  : review["author"],
          "rating"  : review["author_details"]["rating"]
      }
    else:
      return {
          "author"  : review["author"],
      }

  for review in reviews['results']:

    # Consultamos con nuestra colección
    result = reviews_col.get(
      ids=[str(review['id'])],
      include=[]
    )
    # Si no existe, la añade
    if not result['ids'] and review['content']:
      ids.append(review['id'])
      metadatas.append(get_metadata(review))

      content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                        contents=review['content']).embeddings[0]
      embeddings.append(content_embeddings.values)
      documents.append(review['content'])

      if len(ids) > 0:
        reviews_col.add(
          ids = ids,
          embeddings = embeddings,
          metadatas = metadatas,
          documents = documents
        )

  print(f"Added {len(ids)} reviews.")

# Ejemplo con la película 99 de TMDB (Todo Sobre Mi Madre)
add_reviews_to_collection("99")


Added 4 reviews.


## Añadir actores a la colección "people" de ChromaDB

In [11]:
def add_person_to_collection(person_id):
  people_col = chroma_client.get_or_create_collection('people')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  details = get_person_details(person_id)

  # Consultamos con nuestra colección
  result = people_col.get(
    ids=[str(details['id'])],
    include=[]
  )

  if not result['ids'] and details['biography']:
    ids.append(str(details['id']))

    gender = "Not set"
    if details['gender'] == 1:
      gender = "female"
    elif details['gender'] == 2:
      gender = "male"
    elif details['gender'] == 3:
      gender = "non binary"

    metadatas.append({'name': details['name'], 'department': details['known_for_department'], 'gender': gender})
    content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                      contents=details['biography']).embeddings[0]
    embeddings.append(content_embeddings.values)
    documents.append(details['biography'])

    if len(ids) > 0:
      people_col.add(
      ids = ids,
      embeddings = embeddings,
      metadatas = metadatas,
      documents = documents
    )

    print(f"Added {details['name']}")
  else:
    print("Person already exists in collection")

# Ejemplo con persona 31 (Tom Hanks)
add_person_to_collection("31")


Added Tom Hanks


## Ejemplo de consulta a la colección "movies" de ChromaDB

In [12]:
# Ejemplo de hacerle una pregunta a la colección
query = "película sobre coches"

# Hay que hacer un embedding porque hemos no usamos el modelo
# nativo de chromadb, sino el de gemini al meterlos en la colección
query_embedding = client.models.embed_content(model="gemini-embedding-001",
                                              contents=query).embeddings[0]
movies_col = chroma_client.get_or_create_collection('movies')

res = movies_col.query(
    query_embeddings=[query_embedding.values],
    n_results=3
)

res['metadatas'][0]

[{'vote_average': 0.0,
  'popularity': 12.0709,
  'movie_title': 'The Super Mario Galaxy Movie',
  'director': 'Benny Safdie, ',
  'vote_count': 0,
  'release_date': '2026-04-01',
  'cast': 'Chris Pratt, Anya Taylor-Joy, Charlie Day, Jack Black, Keegan-Michael Key, Kevin Michael Richardson, Brie Larson, '},
 {'director': '',
  'vote_count': 1,
  'vote_average': 10.0,
  'movie_title': 'The Super Mario Bros. Movie: Replayed',
  'release_date': '2023-09-23',
  'popularity': 1.3808,
  'cast': 'Chris Pratt, Charlie Day, Jack Black, John DiMaggio, '},
 {'release_date': '2023-04-05',
  'popularity': 22.2602,
  'vote_average': 7.6,
  'cast': 'Chris Pratt, Anya Taylor-Joy, Charlie Day, Jack Black, Keegan-Michael Key, Seth Rogen, Fred Armisen, Sebastian Maniscalco, Charles Martinet, Kevin Michael Richardson, Khary Payton, Rino Romano, John DiMaggio, Jessica DiCicco, Eric Bauza, Juliet Jelenic, Scott Menville, Carlos Alazraqui, Jason Broad, Ashly Burch, Rachel Butera, Cathy Cavadini, Will Collyer

In [13]:
#AGENTE 1: (investigador de cine)
class AgenteInvestigadorCine:
    def __init__(self, tmdb_api_key, gemini_client, chroma_client):
        self.TMDB_HEADERS = {
            "accept": "application/json",
            "Authorization": f"Bearer {tmdb_api_key}" #headers de TMDB
        }
        self.client = gemini_client
        self.chroma_client = chroma_client

    # Buscar películas
    def buscar_peliculas_tmdb(self, titulo):
        url = f"https://api.themoviedb.org/3/search/movie?query={titulo}&include_adult=false&language=en-US&page=1" #crear la url
        response = requests.get(url, headers=self.TMDB_HEADERS)
        return response.json()

    # Obtener reparto y director
    def obtener_reparto_tmdb(self, movie_id):
        url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?language=en-US"
        response = requests.get(url, headers=self.TMDB_HEADERS)
        cast_data = response.json()
        director = ", ".join([p['name'] for p in cast_data['crew'] if p['job']=="Director"]) #Busca el director
        cast = ", ".join([p['name'] for p in cast_data['cast']]) #elenco
        return director, cast

    # Guardar películas en ChromaDB
    #Esta funcion consulta si ya existe la informacion en nuestra base de datos chromaDB y si no existe la crea consultando en TMDB
    def guardar_peliculas_chroma(self, peliculas):
        movies_col = self.chroma_client.get_or_create_collection('movies')
        ids, embeddings, metadatas, documents = [], [], [], []

        for movie in peliculas['results']:
            result = movies_col.get(ids=[str(movie['id'])], include=[])
            if not result['ids'] and movie.get('overview'):
                director, cast = self.obtener_reparto_tmdb(movie['id'])
                content_emb = self.client.models.embed_content(
                    model="gemini-embedding-001",
                    contents=movie['overview']
                ).embeddings[0].values

                ids.append(str(movie['id']))
                embeddings.append(content_emb) #embedings
                metadatas.append({ #metadata guarda director , titulo ...
                    "titulo": movie['title'],
                    "director": director,
                    "cast": cast,
                    "popularidad": movie['popularity'],
                    "fecha_estreno": movie.get('release_date'),
                    "voto_promedio": movie.get('vote_average'),
                    "voto_count": movie.get('vote_count')
                })
                documents.append(movie['overview']) #descripcion de la peli o sinopsis

        if ids:
            movies_col.add(ids=ids, embeddings=embeddings, metadatas=metadatas, documents=documents)
        print(f"Agregado(s) {len(ids)} película(s) a ChromaDB")


In [15]:
#Crear agente
agente = AgenteInvestigadorCine(TMDB_API_KEY, client, chroma_client)

# Buscar y guardar películas
pelis = agente.buscar_peliculas_tmdb("Mario")
agente.guardar_peliculas_chroma(pelis)


Agregado(s) 14 película(s) a ChromaDB
